<a href="https://colab.research.google.com/github/fernandolievano/procesamiento-del-habla/blob/main/%5BPH%5D_TP3_chatbot_retrieval_template_parte_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TP3 Chatbots basados en recuperación de la información

En inglés information retrieval chatbots

# Motor de búsqueda

* Búsqueda por palabras clave: Extrae palabras clave de la pregunta del usuario y busca coincidencias en las preguntas almacenadas.

* Similitud del coseno: Si has representado las preguntas como vectores (por ejemplo, usando TF-IDF o word embeddings), puedes usar la similitud del coseno para medir la distancia entre las preguntas.

* Embeddings: Utiliza modelos de word embeddings como por ejemplo Word2Vec para obtener representaciones semánticas de las preguntas y las consultas del usuario.

## Librerías

Debes trabajar en Python. Puedes usar las librerías sklearn, pandas, spacy o nltk o gensim para el punto de usar o buscar embeddings.

In [2]:
!pip install spacy --quiet
!python -m spacy download es_core_news_md --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 MB 18.1 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('es_core_news_md')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd
import numpy as np
import spacy

### Utils y Setup

In [4]:
def print_title(title):
    print('===' * 30)
    print(f'🔷 {title}')
    print('===' * 30)
    print('\n')

In [5]:
# Carga del modelo spaCy (md)
nlp = spacy.load('es_core_news_md')

> Se eligió el modelo `es_core_news_md` porque el modelo md es más potente y nos va a permitir mejorar la calidad de las respuestas del chatbot.

## Actividades



### 1) Elaborar un dataset de preguntas y respuestas para crear un Chatbot para un aplicación particular. ( 3 puntos )

1.1 Debe definir la aplicación (atención al cliente bancario, atención a estudiantes universitarios, etc).

1.2 El listado de preguntas y respuestas debe tener como mínimo 20 elementos pregunta - respuesta.

###  2) Crear el chatbot utilizando TFIDF y similitud del coseno. (1 punto)

### 3) Crear otro chatbot utilizando embeddings. Indique cuál embedding (1 punto) pre-entrenado eligió.

### 4) Muestra ambos chatbots funcionando (1 punto)

Adjuntar la lista de preguntas y respuestas utilizadas para probar el funcionamiento.

Releve o indique cuáles respondió correctamente y cuáles no.

### 5) Añade tus conclusiones de todo lo realizado (2 punto)

* Resalte e indique en cuáles respuestas falla o tiene problemas.
* Cuáles preguntas confunde.
* Compare los resultados de los chatbots.



### No olvides:

* Explicar tus decisiones y configuraciones. Añadir tus conclusiones.
* Anunciar en el foro cuál será tu aplicación y postear tu entrega y tus avances.
* Debes subir tu notebook a un repo GitHub público de tu propiedad compartido + enlace colab.
* Documentar todo el proceso.
* Citar tus fuentes





# 1. Elaboración de dataset de preguntas y respuestas para crear un Chatbot para una aplicación particular.

### 1.1 Definición de aplicación

Se eligió trabajar en un **chatbot de atención al cliente para una tienda online** debido a que este tipo de apps presentan consultas frecuentes y variadas relacionadas con envíos, pagos, devoluciones, etc.

Además, trabajar con una tienda online nos resulta útil para compara distintos enfoques de chatbot ya que los usuarios pueden realizar la misma consulta usando palabras diferentes, o expresiones similares.

### 1.2 Listado de preguntas y respuestas

| Pregunta                                  | Respuesta                                                                                                |
| ----------------------------------------- | -------------------------------------------------------------------------------------------------------- |
| ¿Hacen envíos a todo el país?             | Sí, realizamos envíos a todo el país mediante correo privado.                                            |
| ¿Cuánto tarda en llegar mi compra?        | El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles.                   |
| ¿Qué métodos de pago aceptan?             | Aceptamos tarjetas de crédito, débito, transferencias y billeteras virtuales.                            |
| ¿Puedo pagar en cuotas?                   | Sí, algunas tarjetas permiten pagar en cuotas sin interés.                                               |
| ¿Cómo puedo seguir mi pedido?             | Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico.             |
| ¿Tienen retiro en sucursal?               | Sí, ofrecemos retiro en sucursal sin costo adicional.                                                    |
| ¿Qué pasa si el producto llega dañado?    | Si el producto llega dañado, podés solicitar un cambio o devolución dentro de las primeras 48 horas.     |
| ¿Puedo devolver un producto?              | Sí, aceptamos devoluciones dentro de los 10 días posteriores a la compra.                                |
| ¿Los productos tienen garantía?           | Sí, todos nuestros productos cuentan con garantía oficial del fabricante.                                |
| ¿Cómo creo una cuenta?                    | Podés registrarte desde el botón “Crear cuenta” completando tus datos personales.                        |
| Olvidé mi contraseña, ¿qué hago?          | Podés recuperar tu contraseña desde la opción “Olvidé mi contraseña” en la pantalla de inicio de sesión. |
| ¿Hay descuentos disponibles?              | Sí, regularmente ofrecemos promociones y descuentos especiales en distintos productos.                   |
| ¿Cómo aplico un cupón de descuento?       | El cupón se puede ingresar durante el proceso de pago antes de finalizar la compra.                      |
| ¿Tienen stock disponible?                 | El stock actualizado aparece en la página de cada producto.                                              |
| ¿Puedo cancelar una compra?               | Sí, podés cancelar tu compra antes de que el pedido sea despachado.                                      |
| ¿Qué hago si mi pedido no llegó?          | Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.                |
| ¿Venden productos originales?             | Sí, trabajamos únicamente con productos originales y oficiales.                                          |
| ¿Cómo contacto con atención al cliente?   | Podés contactarnos por correo electrónico, teléfono o chat online.                                       |
| ¿El sitio es seguro para comprar?         | Sí, utilizamos protocolos de seguridad y cifrado para proteger tus datos.                                |
| ¿Puedo modificar la dirección de entrega? | Sí, podés modificar la dirección antes de que el pedido sea enviado.                                     |


In [6]:
data = [
    {
        "pregunta": "¿Hacen envíos a todo el país?",
        "respuesta": "Sí, realizamos envíos a todo el país mediante correo privado."
    },
    {
        "pregunta": "¿Cuánto tarda en llegar mi compra?",
        "respuesta": "El tiempo de entrega depende de tu ubicación, pero suele ser entre 3 y 7 días hábiles."
    },
    {
        "pregunta": "¿Qué métodos de pago aceptan?",
        "respuesta": "Aceptamos tarjetas de crédito, débito, transferencias y billeteras virtuales."
    },
    {
        "pregunta": "¿Puedo pagar en cuotas?",
        "respuesta": "Sí, algunas tarjetas permiten pagar en cuotas sin interés."
    },
    {
        "pregunta": "¿Cómo puedo seguir mi pedido?",
        "respuesta": "Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico."
    },
    {
        "pregunta": "¿Tienen retiro en sucursal?",
        "respuesta": "Sí, ofrecemos retiro en sucursal sin costo adicional."
    },
    {
        "pregunta": "¿Qué pasa si el producto llega dañado?",
        "respuesta": "Si el producto llega dañado, podés solicitar un cambio o devolución dentro de las primeras 48 horas."
    },
    {
        "pregunta": "¿Puedo devolver un producto?",
        "respuesta": "Sí, aceptamos devoluciones dentro de los 10 días posteriores a la compra."
    },
    {
        "pregunta": "¿Los productos tienen garantía?",
        "respuesta": "Sí, todos nuestros productos cuentan con garantía oficial del fabricante."
    },
    {
        "pregunta": "¿Cómo creo una cuenta?",
        "respuesta": "Podés registrarte desde el botón 'Crear cuenta' completando tus datos personales."
    },
    {
        "pregunta": "Olvidé mi contraseña, ¿qué hago?",
        "respuesta": "Podés recuperar tu contraseña desde la opción 'Olvidé mi contraseña' en la pantalla de inicio de sesión."
    },
    {
        "pregunta": "¿Hay descuentos disponibles?",
        "respuesta": "Sí, regularmente ofrecemos promociones y descuentos especiales en distintos productos."
    },
    {
        "pregunta": "¿Cómo aplico un cupón de descuento?",
        "respuesta": "El cupón se puede ingresar durante el proceso de pago antes de finalizar la compra."
    },
    {
        "pregunta": "¿Tienen stock disponible?",
        "respuesta": "El stock actualizado aparece en la página de cada producto."
    },
    {
        "pregunta": "¿Puedo cancelar una compra?",
        "respuesta": "Sí, podés cancelar tu compra antes de que el pedido sea despachado."
    },
    {
        "pregunta": "¿Qué hago si mi pedido no llegó?",
        "respuesta": "Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo."
    },
    {
        "pregunta": "¿Venden productos originales?",
        "respuesta": "Sí, trabajamos únicamente con productos originales y oficiales."
    },
    {
        "pregunta": "¿Cómo contacto con atención al cliente?",
        "respuesta": "Podés contactarnos por correo electrónico, teléfono o chat online."
    },
    {
        "pregunta": "¿El sitio es seguro para comprar?",
        "respuesta": "Sí, utilizamos protocolos de seguridad y cifrado para proteger tus datos."
    },
    {
        "pregunta": "¿Puedo modificar la dirección de entrega?",
        "respuesta": "Sí, podés modificar la dirección antes de que el pedido sea enviado."
    }
]

print_title('Dataset de preguntas y respuestas')
df = pd.DataFrame(data)

df.sample(5)

🔷 Dataset de preguntas y respuestas




,pregunta,respuesta
15,¿Qué hago si mi pedido no llegó?,"Si tu pedido no llegó en la fecha estimada, po..."
8,¿Los productos tienen garantía?,"Sí, todos nuestros productos cuentan con garan..."
13,¿Tienen stock disponible?,El stock actualizado aparece en la página de c...
18,¿El sitio es seguro para comprar?,"Sí, utilizamos protocolos de seguridad y cifra..."
11,¿Hay descuentos disponibles?,"Sí, regularmente ofrecemos promociones y descu..."


# 2. Chatbot TFIDF + Similitud de coseno


In [7]:
print_title('TF-IDF')

vectorizer = TfidfVectorizer()
tf_idf_matrix = vectorizer.fit_transform(df['pregunta'])

🔷 TF-IDF




In [8]:
print_title('Chatbot TFIDF')

def chatbot_tfidf(question):
    # primero, transformamos el mensaje del usuario
    message_vector = vectorizer.transform([question])
    # en segundo lugar, calculamos la similitud
    similarities = cosine_similarity(message_vector, tf_idf_matrix)
    # obtenemos el índice más parecido
    index = similarities.argmax()
    # obtenemos el score
    score = similarities[0, index]
    # configuramos un umbral mínimo para responder
    if score < 0.2:
        # si el puntaje es muy bajo, respondemos con un mensaje genérico
        return 'Lo siento, no entendí tu consulta'
    # caso contrario,  retornamos respuesta
    return df.iloc[index]['respuesta']

🔷 Chatbot TFIDF




# 3. Chatbot Embeddings

In [9]:
print_title('Generación de embeddings')

vectors = []

for question in df['pregunta']:
    # procesamos cada pregunta y la convertimos en un objeto Doc
    doc = nlp(question)
    # extraemos cada vector del doc actual, y lo añadimos a nuestra lista
    vectors.append(doc.vector)
# convertimos en un array de numpy para
# tener operaciones numéricas más eficientes
vectors = np.array(vectors)
print(f'VECTORES: \n{vectors}')

print(f'FORMA: {vectors.shape}')

🔷 Generación de embeddings


VECTORES: 
[[ 0.5419376   0.04616624 -0.12918621 ...  1.8022925  -0.32414982
  -2.2396789 ]
 [ 0.70963144 -1.0796876   0.5304512  ...  1.2002162  -0.44696754
  -2.5408664 ]
 [ 0.14976428  0.32909468  1.4299     ...  1.2966999  -0.81720144
  -3.4422772 ]
 ...
 [ 0.47268876  0.07235998  0.07485747 ...  0.6252163  -0.8192023
  -3.041854  ]
 [-1.6470549  -0.38719997 -0.47615612 ... -0.05082525  0.4005188
  -2.929781  ]
 [ 0.01361129 -0.96668094  2.1605258  ...  0.72660995 -1.2920713
  -2.741057  ]]
FORMA: (20, 300)


In [10]:
print_title('Chatbot Embeddings')

def chatbot_embeddings(question):
    # transformamos el mensaje del usuario a vector
    message_vector = nlp(question).vector.reshape(1, -1)
    # calculamos la similitud
    similarities = cosine_similarity(message_vector, vectors)
    # obtenemos el índice más similar
    index = similarities.argmax()
    # obtenemos el score
    score = similarities[0, index]
    # configuramos un umbral mínimo para responder
    if score < 0.2:
        # si el puntaje es demasiado bajo, no respondemos
        return 'Lo siento, no entendí tu pregunta.'
    # retornamos la respuesta
    return df.iloc[index]['respuesta']

🔷 Chatbot Embeddings




# 4. Mostrar ambos chatbots funcionando

## Test chatbot TFIDF

In [11]:
print_title('Testing chatbot')

while True:
    question = input('Pregunta: ')
    if question.lower() == 'salir':
        break
    answer = chatbot_tfidf(question)
    print('Respuesta:', answer)

🔷 Testing chatbot


Pregunta: Hola, puedo pagar con tarjeta de crédito?
Respuesta: Sí, algunas tarjetas permiten pagar en cuotas sin interés.
Pregunta: y puedo cancelar una compra?
Respuesta: Sí, podés cancelar tu compra antes de que el pedido sea despachado.
Pregunta: y como se si mi pedido ya fue despachado?
Respuesta: Si tu pedido no llegó en la fecha estimada, podés comunicarte con soporte para revisarlo.
Pregunta: puedo seguir el envío de alguna manera?
Respuesta: Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico.
Pregunta: salir


## Test chatbot Embeddings

In [12]:
print_title('Testing chatbot')

while True:
    question = input('Pregunta: ')
    if question.lower() == 'salir':
        break
    answer = chatbot_embeddings(question)
    print('Respuesta:', answer)

🔷 Testing chatbot


Pregunta: Hola, quiero consultar sobre el stock de un producto
Respuesta: El cupón se puede ingresar durante el proceso de pago antes de finalizar la compra.
Pregunta: Puedo saber el precio del producto?
Respuesta: El cupón se puede ingresar durante el proceso de pago antes de finalizar la compra.
Pregunta: Puedo seguir el envío de mi pedido?
Respuesta: Una vez despachado el pedido, te enviaremos un código de seguimiento por correo electrónico.
Pregunta: Puedo cancelar una compra?
Respuesta: Sí, podés cancelar tu compra antes de que el pedido sea despachado.
Pregunta: y puedo pagar con tarjeta de crédito?
Respuesta: Sí, podés modificar la dirección antes de que el pedido sea enviado.
Pregunta: no, hablaba sobre el método de pago
Respuesta: Sí, realizamos envíos a todo el país mediante correo privado.
Pregunta: Quiero pagar con efectivo
Respuesta: Podés contactarnos por correo electrónico, teléfono o chat online.
Pregunta: salir


# 5. Conclusiones

### Análisis y Conclusiones de los Chatbots (TF-IDF vs. Embeddings)

#### Chatbot TF-IDF
**Fortalezas:**
*   **Relevancia en respuestas directas:** Responde con precisión a preguntas claras y con términos clave directos.
*   **Buena identificación de la intención:** Demuestra capacidad para encontrar contenido relevante basado en palabras clave, incluso con ligeras variaciones.

**Debilidades:**
*   **Respuestas a veces imprecisas:** Puede dar información relacionada pero no exactamente lo que se busca si la pregunta tiene matices muy específicos que no se capturan por las palabras clave.

#### Chatbot Embeddings (usando `es_core_news_md`)
**Fortalezas:**
*   **Manejo de sinónimos y variaciones:** Logra identificar la intención en preguntas con formulaciones variadas, lo que indica que captura similitudes semánticas.

**Debilidades:**
*   **Fallos en la comprensión contextual:** Presenta serios problemas para entender la intención del usuario, ofreciendo respuestas irrelevantes en muchos casos (ej. confundiendo stock o precio con cupones, o métodos de pago con direcciones de envío).
*   **Sensibilidad a la formulación de preguntas:** Se desorienta cuando la pregunta se aleja de las formulaciones exactas del dataset, asociándola a vectores semánticos incorrectos.

#### Comparación General

Para este conjunto de datos, el **chatbot TF-IDF se mostró más robusto y consistente** en la recuperación de respuestas relevantes. Aunque los embeddings tienen el potencial de comprender el significado semántico, la implementación actual con `es_core_news_md` falló en mantener el contexto y la relevancia en preguntas con pequeñas variaciones o que requerían una comprensión más profunda.

# Referencias

### Uso de IA:
- [Generación de preguntas y respuestas para el dataset](https://chatgpt.com/share/6a0bed2e-7eb4-83e9-89f8-5c4d8921726b)


